# Alcohol product data: barcode matching

Evaluate candidate matches from a pinned sample of beer, wine and spirits records. Keep identifier evidence and source links instead of assuming one barcode means one product.

This notebook builds a 100-approval local exercise from the August 20, 2026 source snapshot. Read [README.md](README.md) for the workflow and current sample-data terms. The MIT license covers code only; it does not grant redistribution or commercial data rights.


In [ ]:
import json
import os
from pathlib import Path
from workflow import build, lookup, normalize_gtin

archive = Path(os.environ.get("COLA_SAMPLE_ARCHIVE", ".cache/cola-sample-pack-v2.zip"))
output = Path(os.environ.get("COLA_SAMPLE_OUTPUT", "output"))
rows, report = build(archive, output)
print(json.dumps(report, indent=2))

## Inspect evidence

The exercise intentionally contains 60 rows with supported numeric GTIN evidence and 40 without it. Wine, malt beverages and spirits are all included. This is not population sampling.

In [ ]:
assert len(rows) == 100
assert len({r["ttb_id"] for r in rows}) == 100
positive = next(row for row in rows if json.loads(row["gtins_json"]))
code = json.loads(positive["gtins_json"])[0]
candidates = lookup(rows, code)
print("GTIN:", code)
print("Candidate approval IDs:", [r["ttb_id"] for r in candidates])
print("Evidence:", positive["barcode_evidence_json"])
print("TTB source:", positive["source_ttb_url"])
assert positive["ttb_id"] in {r["ttb_id"] for r in candidates}

## Invalid inputs and sample-only misses

A checksum rejects many transcription mistakes but cannot verify GS1 assignment or real product identity. Preserve leading zeros. Missing candidates in this teaching subset do not establish missing records in the full registry.

In [ ]:
assert normalize_gtin("036000291452") == normalize_gtin("0036000291452")
try:
    normalize_gtin("036000291453")
except ValueError as error:
    print("Rejected invalid example:", error)
else:
    raise AssertionError("Invalid check digit accepted")

controls = [r for r in rows if not json.loads(r["gtins_json"])]
assert len(controls) == 40
print("Approvals without supported GTIN in the exercise:", len(controls))

## A bounded, reproducible observation

Compare all extracted code types with supported numeric GTIN evidence in the **original 1,000-row pack**, not the designed 100-row subset.

In [ ]:
any_code = report["approvals_with_any_extracted_code"]
supported = report["approvals_with_supported_valid_gtin"]
count = report["source_approval_rows"]
print(f"In this {count}-approval snapshot: {any_code} approvals have any extracted code; {supported} have a supported valid UPC-A/EAN-13.")
print("Difference:", any_code - supported)
print("Source date:", report["source_generated_at"])
print("Source SHA-256:", report["source_sha256"])

## Optional current-data follow-up

Use the actual example GTIN above with COLA Cloud’s account-backed `search_colas` and `get_cola` tools. Ask for every candidate and inspect evidence. The notebook makes no API or MCP calls and spends no account quota. See README.md for setup and limits.